# Master Daily Orchestration


In [0]:
# =============================================================================
# Notebook:   master_daily.py
# Schedule:   Daily at 23:00
#
# Purpose:
#   Orchestrates the daily EA real-time archive ingest pipeline.
#   Runs the archive ingest notebook, which in turn calls the measure
#   register notebook if new measures are detected.
#
# HOW TO SCHEDULE IN DATABRICKS:
#   1. Open this notebook in the Databricks workspace
#   2. Click "Schedule" in the top right
#   3. Set to daily at 23:00
#   4. Ensure the cluster is set to your DASH cluster
#   5. Save
#
# Exit values (from 10_ea_rt_daily_archive):
#   "success" -- archive ingest completed successfully
#   "no_file" -- the day's archive file was not available (HTTP 404); nothing
#                was landed, so this run fails and should be re-run later
#   Any other value indicates a failure in a child notebook.
#
# Child notebooks are called by relative path, so the folder can be anywhere.
#
# Step 2 (21_fgs_constituency_exposure) does not depend on step 1, so it runs
# even if the archive is not yet available. Failures are collected and raised at
# the end, so the run is still marked as failed.
#
# Job numbering:
#   10_ea_rt_daily_archive  -- downloads yesterday's archive CSV,
#                              lands to Bronze, updates measure register
#                              if new measures detected
#   21_fgs_constituency_exposure -- days each constituency was forecast at each
#                              risk rank, by month and flood source
#   26_fgs_forecast_verification -- forecast risk against the alerts and warnings
#                              in force, by constituency, date and lead day
# =============================================================================
# =============================================================================
# STEP 1: EA REAL-TIME DAILY ARCHIVE INGEST
# =============================================================================

print("=" * 60)
print("STEP 1: EA real-time daily archive ingest")
print("=" * 60)

errors         = []
archive_result = "failed"

try:
    archive_result = dbutils.notebook.run(
        path            = "../jobs/10_ea_rt_daily_archive",
        timeout_seconds = 7200,  # 2 hours -- allows for slow EA API and measure register enrichment
        arguments       = {}
    )
    print(f"Archive ingest result: {archive_result}")
    if archive_result != "success":
        errors.append(f"Daily archive ingest failed with: {archive_result}")
except Exception as e:
    errors.append(f"Daily archive ingest: {e}")
    print(f"ERROR in step 1: {e}")

# COMMAND ----------

# =============================================================================
# STEP 2: FGS CONSTITUENCY EXPOSURE
# Full rebuild from the statement history. Independent of step 1.
# =============================================================================

print("=" * 60)
print("STEP 2: FGS constituency exposure")
print("=" * 60)

exposure_result = "failed"

try:
    exposure_result = dbutils.notebook.run(
        path            = "../jobs/21_fgs_constituency_exposure",
        timeout_seconds = 1800,   # 30 minutes
        arguments       = {}
    )
    if exposure_result != "success":
        errors.append(f"FGS exposure returned unexpected value: {exposure_result}")
except Exception as e:
    errors.append(f"FGS exposure: {e}")
    print(f"ERROR in step 2: {e}")

print(f"FGS exposure result: {exposure_result}")

# COMMAND ----------

# =============================================================================
# STEP 3: FGS FORECAST VERIFICATION
# Forecast risk against the alerts and warnings in force. Reads the warning
# episodes that Master_polling keeps current (job 25). Independent of steps 1 and 2.
# =============================================================================

print("=" * 60)
print("STEP 3: FGS forecast verification")
print("=" * 60)

verification_result = "failed"

try:
    verification_result = dbutils.notebook.run(
        path            = "../jobs/26_fgs_forecast_verification",
        timeout_seconds = 3600,   # 1 hour
        arguments       = {}
    )
    if verification_result != "success":
        errors.append(f"FGS forecast verification returned unexpected value: {verification_result}")
except Exception as e:
    errors.append(f"FGS forecast verification: {e}")
    print(f"ERROR in step 3: {e}")

print(f"FGS forecast verification result: {verification_result}")

# COMMAND ----------

# =============================================================================
# COMPLETE
# =============================================================================

print("=" * 60)
print("Master daily run summary")
print(f"  Archive ingest:  {archive_result}")
print(f"  FGS exposure:    {exposure_result}")
print(f"  FGS verification: {verification_result}")
print("=" * 60)

if errors:
    raise Exception("Daily run had failures: " + " | ".join(errors))